# 🚀 Ready-to-Run: How2Sign Preprocessing V4 on Google Colab GPU
This notebook runs the **high-throughput Preprocessor V4** streamer on Google Colab (T4 / V100 / A100 GPU).

### 🌟 Preprocessor V4 Key Highlights:
- **RTMW / WholeBody 133-Keypoint Extraction**: High-fidelity 2D/3D body, hands, and facial landmarks with CUDA ONNX acceleration.
- **Double-Buffered Producer-Consumer**: Computes on GPU while asynchronously uploading completed `.pt` shards to Google Drive in the background (zero GPU idle latency).
- **EMA Sternum Anchoring**: Removes long-duration body drift while preserving grammatical leans.
- **Frame-Rate Invariant $\Delta t$ Kinematics**: Savitzky-Golay polynomial derivatives normalized by physical timestamp intervals.
- **19D ASL Phonology**: Bounded $[0, 1]$ finger curls normalized by palm length + Gaussian face-contact kernels.
- **Crash-Resilient Ledger**: Automatically skips already processed clips if Colab disconnects.

### Step 1: Mount Google Drive
Mount your Google Drive where How2Sign input videos are stored and where preprocessed shards will be uploaded.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Install High-Speed GPU Dependencies
Uninstalls CPU onnxruntime to avoid library conflicts and installs `onnxruntime-gpu` with CUDA acceleration.

In [ ]:
# 1. Cleanly uninstall both CPU and GPU onnxruntime to avoid library collisions
!pip uninstall -y onnxruntime onnxruntime-gpu

# 2. Install rtmlib with --no-deps so it DOES NOT overwrite GPU build with CPU onnxruntime
!pip install -q --no-deps rtmlib

# 3. Install onnxruntime-gpu and NVIDIA runtime wheels (providing libcublasLt.so.13 & libcudnn.so.9)
!pip install -q onnxruntime-gpu nvidia-cublas nvidia-cudnn-cu12
!pip install -q opencv-python sentence-transformers torch torchvision

# 4. Register pip-installed NVIDIA libraries into Linux system-wide dynamic linker cache
!python3 -c "import site, glob; [print(p) for p in glob.glob(site.getsitepackages()[0] + '/nvidia/*/lib')]" | sudo tee /etc/ld.so.conf.d/nvidia_wheels.conf
!sudo ldconfig

# 5. Restart session programmatically so Python loads the fresh GPU shared libraries
print("[+] Installation complete. Restarting session to load GPU binaries...")
try:
    from google.colab import runtime
    runtime.restart()
except Exception:
    import os
    os.kill(os.getpid(), 9)


### Step 3: Verify GPU & ONNX Runtime CUDA Acceleration

In [ ]:
import torch
import onnxruntime as ort

print("=" * 60)
print(f"[+] PyTorch CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[+] Active GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"[+] Available VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

providers = ort.get_available_providers()
print(f"[+] ONNX Runtime Available Providers: {providers}")

assert "CUDAExecutionProvider" in providers, (
    f"CUDAExecutionProvider not detected! Current providers: {providers}.\n"
    "Please make sure you ran Step 2 and restarted the session (Runtime > Restart session)."
)

# Verify real session instantiation with RTMW on GPU
from rtmlib import Wholebody
print("[+] Testing RTMW WholeBody session on GPU...")
test_model = Wholebody(mode="lightweight", backend="onnxruntime", device="cuda")
active = test_model.pose_model.session.get_providers()
print(f"[+] Active RTMW ONNX Providers: {active}")
assert "CUDAExecutionProvider" in active, f"ERROR: Model fell back to CPU: {active}"
print("[SUCCESS] CUDAExecutionProvider is active and verified on GPU!")
print("=" * 60)


### Step 4: Clone & Configure Repository
Clones/pulls the repository and sets up path mappings so scripts can be called from anywhere.

In [ ]:
import os
import shutil
from pathlib import Path

repo_dir = Path("/content/netmaui-singlanguage")
repo_url = "https://github.com/Codering2012/netmaui-singlanguage.git"

# 1. Clone or pull repository
if not repo_dir.exists():
    print(f"[*] Cloning repository from {repo_url}...")
    !git clone {repo_url} /content/netmaui-singlanguage
else:
    print(f"[*] Pulling latest updates into {repo_dir}...")
    !cd /content/netmaui-singlanguage && git pull

# 2. Ensure /content/Kaggle script resolves directly
nested_folder = repo_dir / "Kaggle script"
link_target = Path("/content/Kaggle script")
if nested_folder.exists() and not link_target.exists():
    !ln -s "/content/netmaui-singlanguage/Kaggle script" "/content/Kaggle script"

# Handle case where repo was cloned directly into /content/Kaggle script
double_nested = Path("/content/Kaggle script/Kaggle script")
if double_nested.exists() and not (link_target / "preprocessing").exists():
    !ln -s "/content/Kaggle script/Kaggle script/preprocessing" "/content/Kaggle script/preprocessing"

%cd /content

# 3. Discover exact streamer script path
candidates = [
    Path("/content/Kaggle script/preprocessing/how2sign_colab_v4_streamer.py"),
    Path("/content/Kaggle script/Kaggle script/preprocessing/how2sign_colab_v4_streamer.py"),
    Path("/content/netmaui-singlanguage/Kaggle script/preprocessing/how2sign_colab_v4_streamer.py"),
]
streamer_script = next((p for p in candidates if p.exists()), None)
assert streamer_script is not None, "Error: Could not locate how2sign_colab_v4_streamer.py! Check repository clone."
print(f"[+] Verified Preprocessor V4 streamer script at: {streamer_script}")

### Step 5: Directory Preparation & Health Check
Checks for input How2Sign archives/videos on Google Drive and prepares local scratch and output directories.

In [ ]:
from pathlib import Path

# Auto-detect How2Sign directory on Google Drive
candidate_dirs = [
    Path("/content/drive/MyDrive/train how2sign"),
    Path("/content/drive/MyDrive/How2Sign"),
    Path("/content/drive/MyDrive/train_how2sign"),
    Path("/content/drive/MyDrive/how2sign"),
]
drive_dir = next((d for d in candidate_dirs if d.exists()), candidate_dirs[0])
output_drive_dir = Path("/content/drive/MyDrive/How2Sign_Preprocessed_V4")
scratch_dir = Path("/content/scratch")

output_drive_dir.mkdir(parents=True, exist_ok=True)
scratch_dir.mkdir(parents=True, exist_ok=True)

print(f"[*] Detected Input Drive Directory: {drive_dir}")
if drive_dir.exists():
    zip_files = list(drive_dir.glob("*.zip"))
    z_part_files = list(drive_dir.glob("*.z*"))
    mp4_files = list(drive_dir.rglob("*.mp4"))
    csv_files = list(drive_dir.glob("*.csv"))
    print(f"[+] Found {len(zip_files)} .zip archives, {len(z_part_files)} split volumes, {len(mp4_files)} .mp4 videos, {len(csv_files)} .csv files.")
    for z in zip_files:
        print(f"    - Archive: {z.name}")
else:
    print(f"[!] Warning: {drive_dir} not found. Please verify your Google Drive folder path.")

print(f"[+] Output Drive Directory: {output_drive_dir} (Ready)")
print(f"[+] Scratch Directory: {scratch_dir} (Ready)")

### Step 6: Launch How2Sign Preprocessor V4 Streamer
Runs double-buffered GPU streaming with ping-pong scratch memory and asynchronous background Drive uploads.

In [ ]:
# Execute the production How2Sign V4 double-buffered streamer targeting segmented clips
# Prepend NVIDIA wheel paths to LD_LIBRARY_PATH and stream with real-time per-clip progress
!LD_LIBRARY_PATH=$(python3 -c "import site, glob; print(':'.join(glob.glob(site.getsitepackages()[0] + '/nvidia/*/lib')))"):"$LD_LIBRARY_PATH" \
python "{streamer_script}" \
    --drive-dir "{drive_dir}" \
    --output-drive-dir "/content/drive/MyDrive/How2Sign_Preprocessed_V4" \
    --local-scratch "/content/scratch" \
    --archive-pattern "train_rgb_front_clips.zip" \
    --chunk-size 100 \
    --backend rtmw \
    --target-fps 30.0


### Step 7: Shard & Ledger Inspection
Verifies completed `.pt` shards on Google Drive, inspects ledger progress, and checks tensor shapes.

In [ ]:
import json
import torch
from pathlib import Path

out_dir = Path("/content/drive/MyDrive/How2Sign_Preprocessed_V4")
shards = sorted(list(out_dir.glob("shard_*.pt")))
ledger_path = out_dir / "ledger.json"

print("=" * 60)
print(f"[+] Total Completed Shards on Drive: {len(shards)}")
if ledger_path.exists():
    with open(ledger_path, "r", encoding="utf-8") as f:
        ledger = json.load(f)
    print(f"[+] Total Processed Clips: {len(ledger.get('completed_clips', []))}")
    print(f"[+] Last Updated: {ledger.get('last_updated', 'N/A')}")

if shards:
    sample_shard = shards[-1]
    print(f"\n[*] Inspecting latest shard: {sample_shard.name} ({sample_shard.stat().st_size / 1e6:.2f} MB)")
    data = torch.load(sample_shard, map_location="cpu")
    print(f"[+] Clips in shard: {len(data)}")
    if len(data) > 0:
        clip = data[0]
        print(f"    - ID: {clip.get('id', 'N/A')}")
        print(f"    - Text: {clip.get('text', clip.get('label', 'N/A'))}")
        for key in ["features", "kinematics", "phonology", "cranial_imu", "quality_weights"]:
            if key in clip:
                val = clip[key]
                shape = getattr(val, "shape", len(val))
                print(f"    - {key}: {shape}")
print("=" * 60)